In [4]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from imblearn.over_sampling import SMOTE

# Load original dataset
df = pd.read_csv('../data/bnpl_applicants.csv')

# Select features
FEATURES = [
    'age',
    'monthly_income',
    'bureau_score',
    'existing_loans',
    'dti_ratio',
    'tenure_months',
    'order_value',
    'on_time_pct_hist',
    'device_tenure_days',
    'app_velocity_30d',
    'city_tier'
]

X = df[FEATURES]
y = df['default']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Preprocessing
pre = ColumnTransformer([
    (
        'num',
        StandardScaler(),
        [c for c in FEATURES if c != 'city_tier']
    ),
    (
        'cat',
        OneHotEncoder(drop='first'),
        ['city_tier']
    )
])

# Fit training data
X_train_pre = pre.fit_transform(X_train)

# Transform test data
X_test_pre = pre.transform(X_test)

# Apply SMOTE only to training data
smote = SMOTE(
    sampling_strategy=0.35,
    random_state=42
)

X_train_bal, y_train_bal = smote.fit_resample(
    X_train_pre,
    y_train
)

# Save preprocessor
os.makedirs('../models', exist_ok=True)

joblib.dump(
    pre,
    '../models/preprocessor.pkl'
)

# Verify variables
print("STEP 3 VARIABLES RESTORED SUCCESSFULLY!")

print("X_train_bal:", X_train_bal.shape)
print("y_train_bal:", y_train_bal.shape)
print("X_test_pre:", X_test_pre.shape)
print("y_test:", y_test.shape)

print("Training default rate:",
      round(y_train_bal.mean() * 100, 2), "%")

print("Testing default rate:",
      round(y_test.mean() * 100, 2), "%")


STEP 3 VARIABLES RESTORED SUCCESSFULLY!
X_train_bal: (29146, 12)
y_train_bal: (29146,)
X_test_pre: (6000, 12)
y_test: (6000,)
Training default rate: 25.92 %
Testing default rate: 10.03 %


In [8]:
import os
import joblib

# Create models folder if it does not exist
os.makedirs("../models", exist_ok=True)

# Check that the trained model exists
try:
    model
except NameError:
    raise RuntimeError(
        "XGBoost model is not trained in this notebook. "
        "Run the model training cell first."
    )

# Save the trained model
model_path = os.path.abspath("../models/xgb_pd_model.pkl")

joblib.dump(model, model_path)

# Verify saved file
if os.path.isfile(model_path):
    print("SUCCESS: XGBoost model saved!")
    print("Model location:", model_path)
    print("File size:", round(os.path.getsize(model_path) / 1024, 2), "KB")
else:
    raise RuntimeError("Model saving failed!")

SUCCESS: XGBoost model saved!
Model location: D:\BNPL RISK ENGIN\bnpl-risk-engine\models\xgb_pd_model.pkl
File size: 0.77 KB
